# Data Cleansing


Yangshunshun (Luna) Lu

The code does data cleansing that prepare data for textual anlysis, visualization and EDA

No AI used

In [1]:
import os
import pandas as pd
import numpy as np
os.environ["PYTHONIOENCODING"] = "utf-8"
os.environ["PYTHONUTF8"] = "1"

In [6]:
def smart_convert_datetime(val):
    if pd.isna(val) or val == "":
        return pd.NaT
    try:

        num_val = pd.to_numeric(val, errors='raise')

        if num_val > 1e9:
            return pd.to_datetime(num_val, unit='ms')
    except:
        pass
    
    return pd.to_datetime(val, errors='coerce')

def clean_and_filter_lyrics_data(input_filepath, output_filepath):

    print(f"loading data: {input_filepath} ...")
    try:
        df = pd.read_csv(input_filepath, encoding='utf-8')
    except UnicodeDecodeError:
        df = pd.read_csv(input_filepath, encoding='latin1')
        
    print(f"initial data shape {df.shape}")
    
    df = df.dropna(axis=1, how='all')
    df = df.loc[:, ~df.columns.str.contains('^Unnamed')]
    
    initial_rows = df.shape[0]
    df = df.drop_duplicates(subset=['track_id'])
    print(f"deleted {initial_rows - df.shape[0]} repeated records")
    
    text_columns = ['artist_name', 'track_name', 'source']
    for col in text_columns:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip()
            
    if 'lyrics' in df.columns and 'lyrics_status' in df.columns:

        df = df.dropna(subset=['lyrics'])

        df = df[df['lyrics_status'] == 'ok']
        
        df = df[~df['lyrics'].astype(str).str.contains('No Lyrics Available', case=False, na=False)]
        df = df[df['lyrics'].astype(str).str.len() > 10]  
    else:
        print("error: no 'lyrics' or 'lyrics_status'")
    
    print(f"remaining data: {df.shape[0]}")
    
    audio_features = [
        'danceability', 'energy', 'loudness', 'speechiness', 
        'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo'
    ]
    existing_features = [col for col in audio_features if col in df.columns]
    for col in existing_features:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        
    if existing_features:
        df = df.dropna(subset=existing_features)
        
    if 'fetched_at' in df.columns:
        print("converting 'fetched_at' to datetime...")
        df['fetched_at'] = df['fetched_at'].apply(smart_convert_datetime)
        
        valid_dates = df['fetched_at'].dropna()
        if not valid_dates.empty:
            print(f"success! from {valid_dates.min()} to {valid_dates.max()}")
        else:
            print("error: all timestamps are invalid after conversion.")
        
    df = df.reset_index(drop=True)
    
    print(f"Cleansing complete! final shape: {df.shape}")
    
    df.to_csv(output_filepath, index=False, encoding='utf-8-sig')
    print(f"data before tokenization saved as: {output_filepath}")
    
    return df

if __name__ == "__main__":
    import os
    input_file = '../Data/covered_with_lyrics_progress.csv'  
    
    # Define and create the directory FIRST
    out_dir = os.path.join('..', 'Data', 'cleaned_data')
    os.makedirs(out_dir, exist_ok=True)
    
    # Set the output path to the NEW directory
    output_file = os.path.join(out_dir, 'cleaned_lyrics_data.csv') 
    
    try:
        cleaned_df = clean_and_filter_lyrics_data(input_file, output_file)
        print(f"Success! File saved to: {output_file}")
    except Exception as e:
        print(f"Error during cleaning: {e}")

loading data: ../Data/covered_with_lyrics_progress.csv ...


C:\Users\Luna_\AppData\Local\Temp\ipykernel_50876\2088370484.py:21: DtypeWarning: Columns (8,9,10,11,12,28,29,30) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(input_filepath, encoding='latin1')


initial data shape (75229, 31)
deleted 2577 repeated records
remaining data: 1075
converting 'fetched_at' to datetime...
success! from 2025-04-23 23:13:20 to 2025-11-12 23:26:40
Cleansing complete! final shape: (1075, 23)
data before tokenization saved as: ..\Data\cleaned_data\cleaned_lyrics_data.csv
Success! File saved to: ..\Data\cleaned_data\cleaned_lyrics_data.csv


In [3]:
#!pip install langdetect nltk

In [7]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem.snowball import SnowballStemmer

try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')

def fix_mojibake(text):

    if not isinstance(text, str):
        return ""
    try:
        return text.encode('latin1').decode('utf-8')
    except (UnicodeEncodeError, UnicodeDecodeError):
        return text

def preprocess_spanish_lyrics(text, stemmer, stop_words):

    text = fix_mojibake(text)
    
    text = text.lower()
    
    text = re.sub(r'\[.*?\]', ' ', text)
    
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\n', ' ', text)
    
    words = text.split()
    
    cleaned_words = [
        stemmer.stem(word) for word in words 
        if word not in stop_words and len(word) > 1 
    ]
    
    return " ".join(cleaned_words)

if __name__ == "__main__":
    print("loading data...")
    df = pd.read_csv('../Data/cleaned_data/cleaned_lyrics_data.csv')
    
    stop_words = set(stopwords.words('spanish'))

    custom_stopwords = {'pa', 'pal', 'ay', 'yeah', 'oh', 'ah', 'eh', 'mami', 'papi'} 
    stop_words = stop_words.union(custom_stopwords)
    
    stemmer = SnowballStemmer('spanish')

    df['processed_lyrics'] = df['lyrics'].apply(lambda x: preprocess_spanish_lyrics(x, stemmer, stop_words))
    

    df = df[df['processed_lyrics'].str.strip() != ""]
    
    out_dir = os.path.join('..', 'Data', 'cleaned_data')
    os.makedirs(out_dir, exist_ok=True)
    
    output_file = os.path.join(out_dir, 'tokenized_lyrics_data.csv')

    df.to_csv(output_file, index=False, encoding='utf-8-sig')
    
    print(f"Completed processing {df.shape[0]} lines.")
    print(f"Tokenized lyrics data saved to: {output_file}")
    

loading data...
Completed processing 1075 lines.
Tokenized lyrics data saved to: ..\Data\cleaned_data\tokenized_lyrics_data.csv
